# 04 — Feature Engineering

**Topics:** Encoding, scaling, binning, interaction terms, date features, target encoding.

**Reference:** [pandas docs](https://pandas.pydata.org/docs/) | [numpy docs](https://numpy.org/doc/stable/)

**Dataset:** Lending Club loan data — credit risk features.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml

loans_raw = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto').frame

# Augment with synthetic features to make exercises richer — do not modify
np.random.seed(42)
n = len(loans_raw)
loans = loans_raw.copy()
loans['loan_date'] = pd.date_range('2018-01-01', periods=n, freq='12H')
loans['annual_income'] = np.random.lognormal(10.8, 0.5, n).round(0)
loans['months_employed'] = np.random.randint(0, 240, n)
loans['credit_score'] = np.random.randint(300, 850, n)
loans['target'] = (loans['class'] == 'good').astype(int)  # 1=good, 0=bad
print(loans.shape)
loans.head()

---
## Exercise 1 — Ordinal & One-Hot Encoding (Manual)

**Task:** Encode categorical features without sklearn — pure pandas.

1. Identify all `object` dtype columns. Print their names and cardinality.
2. One-hot encode `purpose` using `pd.get_dummies`. Use prefix `'purpose'`, drop the first level to avoid multicollinearity. Add the resulting columns to `loans_encoded` (a copy of `loans`).
3. Ordinal-encode `credit_history` manually with this mapping: `{'critical': 0, 'poor': 1, 'good': 2, 'very good': 3, 'perfect': 4}`. If a value doesn't exist in the mapping, assign `-1`.
4. Drop original `purpose` and `credit_history` columns from `loans_encoded`.

In [ ]:
def encode_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    One-hot encode purpose, ordinal encode credit_history.
    Returns loans_encoded with original cols dropped.
    """
    credit_history_map = {'critical': 0, 'poor': 1, 'good': 2, 'very good': 3, 'perfect': 4}
    # YOUR CODE HERE
    pass

loans_encoded = encode_features(loans)

In [ ]:
# --- ASSERTIONS ---
assert 'purpose' not in loans_encoded.columns, "Original purpose column must be dropped"
assert 'credit_history' not in loans_encoded.columns, "Original credit_history must be dropped"
assert 'credit_history_encoded' in loans_encoded.columns or any(c.startswith('credit_history') and 'encoded' in c for c in loans_encoded.columns) or 'credit_history_ord' in loans_encoded.columns, "Need ordinal encoded column"
purpose_cols = [c for c in loans_encoded.columns if c.startswith('purpose_')]
assert len(purpose_cols) > 0, "Must have one-hot encoded purpose columns"
assert loans_encoded[purpose_cols].isin([0, 1]).all().all(), "OHE columns must be 0 or 1"
print(f"✓ Exercise 1 passed — {len(purpose_cols)} purpose dummies created")

---
## Exercise 2 — Scaling (Manual Implementation)

**Task:** Implement 3 scaling methods from scratch using numpy — no sklearn.

1. `min_max_scale(series)`: returns values in [0, 1].
2. `standard_scale(series)`: returns z-scores (mean=0, std=1).
3. `robust_scale(series)`: subtracts median, divides by IQR (Q75 - Q25).

Apply all 3 to `annual_income` and `credit_score`. Return a DataFrame `scaled_df` with 6 columns:
`income_minmax`, `income_standard`, `income_robust`, `score_minmax`, `score_standard`, `score_robust`.

In [ ]:
def min_max_scale(series: pd.Series) -> pd.Series:
    # YOUR CODE HERE
    pass

def standard_scale(series: pd.Series) -> pd.Series:
    # YOUR CODE HERE
    pass

def robust_scale(series: pd.Series) -> pd.Series:
    # YOUR CODE HERE
    pass

def build_scaled_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns DataFrame with 6 scaled columns.
    """
    # YOUR CODE HERE
    pass

scaled_df = build_scaled_df(loans)

In [ ]:
# --- ASSERTIONS ---
expected = ['income_minmax', 'income_standard', 'income_robust', 'score_minmax', 'score_standard', 'score_robust']
assert list(scaled_df.columns) == expected

# MinMax must be in [0, 1]
assert scaled_df['income_minmax'].min() >= 0 and scaled_df['income_minmax'].max() <= 1
assert scaled_df['score_minmax'].min() >= 0 and scaled_df['score_minmax'].max() <= 1

# Standard: mean ~0, std ~1
assert abs(scaled_df['income_standard'].mean()) < 1e-6
assert abs(scaled_df['income_standard'].std() - 1) < 1e-4

# Robust: median ~0
assert abs(scaled_df['income_robust'].median()) < 1e-6
print("✓ Exercise 2 passed")

---
## Exercise 3 — Binning & Custom Buckets

**Task:** Create risk-relevant binned features.

1. Bin `credit_score` into 5 equal-width bins using `pd.cut`. Label them: `'Very Poor'`, `'Poor'`, `'Fair'`, `'Good'`, `'Excellent'`. Assign to column `credit_tier`.
2. Bin `annual_income` into 4 quantile-based bins using `pd.qcut`. Label them: `'Q1'`, `'Q2'`, `'Q3'`, `'Q4'`. Assign to `income_quartile`.
3. Bin `months_employed` into custom intervals: `[0, 12)`, `[12, 60)`, `[60, 120)`, `[120+)`. Label: `'< 1yr'`, `'1-5yr'`, `'5-10yr'`, `'10yr+'`. Assign to `employment_band`.
4. Return a copy of `loans` with these 3 new columns in `loans_binned`.

In [ ]:
def add_binned_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds credit_tier, income_quartile, employment_band to a copy of df.
    """
    # YOUR CODE HERE
    pass

loans_binned = add_binned_features(loans)

In [ ]:
# --- ASSERTIONS ---
for col in ['credit_tier', 'income_quartile', 'employment_band']:
    assert col in loans_binned.columns
    assert loans_binned[col].isna().sum() == 0, f"{col} must have no NaNs"

assert set(loans_binned['credit_tier'].unique()) == {'Very Poor', 'Poor', 'Fair', 'Good', 'Excellent'}
assert set(loans_binned['income_quartile'].unique()) == {'Q1', 'Q2', 'Q3', 'Q4'}
assert set(loans_binned['employment_band'].unique()) == {'< 1yr', '1-5yr', '5-10yr', '10yr+'}

# Quantile bins must be roughly equal-sized
q_counts = loans_binned['income_quartile'].value_counts()
assert q_counts.max() / q_counts.min() < 1.5, "Quantile bins should be roughly equal-sized"

print("✓ Exercise 3 passed")
print(loans_binned[['credit_score', 'credit_tier', 'annual_income', 'income_quartile']].head())

---
## Exercise 4 — Interaction & Polynomial Features (Manual)

**Task:** Create interaction features without sklearn.

Using `loans`, create a new DataFrame `interaction_df` with:
1. `debt_to_income`: `credit_amount / annual_income` (where credit_amount comes from `loans`).
2. `income_x_score`: `annual_income * credit_score` (normalized by dividing each by their mean first).
3. `employment_x_score`: `months_employed * credit_score`.
4. `score_squared`: `credit_score ** 2`.
5. `log_income`: `log(annual_income + 1)`.
6. `log_credit_amount`: `log(credit_amount + 1)` where `credit_amount` is a column in `loans`.

All values should be float64. No loops, no sklearn.

In [ ]:
def build_interaction_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns interaction_df with 6 engineered features.
    """
    # YOUR CODE HERE
    pass

interaction_df = build_interaction_features(loans)

In [ ]:
# --- ASSERTIONS ---
expected = ['debt_to_income', 'income_x_score', 'employment_x_score', 'score_squared', 'log_income', 'log_credit_amount']
for col in expected:
    assert col in interaction_df.columns, f"Missing: {col}"
    assert interaction_df[col].dtype == np.float64 or interaction_df[col].dtype == float, f"{col} must be float64"
    assert interaction_df[col].isna().sum() == 0, f"{col} must have no NaN"

assert (interaction_df['log_income'] >= 0).all(), "log(income+1) must be non-negative"
assert (interaction_df['score_squared'] > 0).all()
print("✓ Exercise 4 passed")

---
## Exercise 5 — Target Encoding (with Leakage Guard)

**Task:** Implement target encoding safely — a technique frequently tested in ML interviews.

Target encoding replaces a categorical value with the mean of the target variable for that category. The risk: data leakage if the current row is included in its own group mean.

1. Implement `target_encode_train(df, col, target_col, smoothing=10)` that:
   - Computes the **smoothed** mean encoding per category: `(n_i * mean_i + smoothing * global_mean) / (n_i + smoothing)` where `n_i` = count of category `i`.
   - Returns a Series with the encoded values.
2. Implement `target_encode_test(test_df, encoding_map)` that applies a precomputed mapping (from training) to a test set, using the global mean for unseen categories.
3. Apply to `purpose` column with `target` as target. Verify that:
   - All encoded values are between 0 and 1.
   - Categories with more 'good' loans have higher encoded values.

In [ ]:
def target_encode_train(df: pd.DataFrame, col: str, target_col: str, smoothing: int = 10):
    """
    Returns (encoded_series, encoding_map_dict)
    encoding_map_dict: {category: smoothed_mean} for use in test set.
    """
    # YOUR CODE HERE
    pass

def target_encode_test(test_df: pd.DataFrame, col: str, encoding_map: dict, global_mean: float) -> pd.Series:
    """
    Apply encoding_map to test set, filling unknowns with global_mean.
    """
    # YOUR CODE HERE
    pass

encoded_purpose, encoding_map = target_encode_train(loans, 'purpose', 'target', smoothing=10)

In [ ]:
# --- ASSERTIONS ---
assert isinstance(encoded_purpose, pd.Series)
assert len(encoded_purpose) == len(loans)
assert encoded_purpose.between(0, 1).all(), "Encoded values must be in [0, 1]"
assert isinstance(encoding_map, dict)
assert all(0 <= v <= 1 for v in encoding_map.values())

# Test set encoding — simulate with a synthetic test
test_sample = pd.DataFrame({'purpose': ['car', 'education', 'UNKNOWN_CATEGORY']})
global_mean = loans['target'].mean()
test_encoded = target_encode_test(test_sample, 'purpose', encoding_map, global_mean)
assert test_encoded.iloc[2] == global_mean, "Unknown category must map to global mean"

print("✓ Exercise 5 passed")
print(pd.Series(encoding_map).sort_values(ascending=False))

---
## Exercise 6 — Date Features Engineering

**Task:** Extract a comprehensive set of date features from `loan_date`.

1. From `loan_date`, extract: `year`, `month`, `day_of_week` (0=Monday), `quarter`, `day_of_year`, `is_weekend` (bool), `days_since_epoch` (days since 2018-01-01).
2. Compute `months_since_loan`: integer months between `loan_date` and a reference date of `2023-01-01`.
3. Add a `season` column: `'Winter'` (Dec–Feb), `'Spring'` (Mar–May), `'Summer'` (Jun–Aug), `'Fall'` (Sep–Nov).
4. Return `loans_dated`: copy of `loans` with all date features added.

In [ ]:
def add_date_features(df: pd.DataFrame, reference_date: str = '2023-01-01') -> pd.DataFrame:
    """
    Add comprehensive date features from loan_date.
    """
    # YOUR CODE HERE
    pass

loans_dated = add_date_features(loans)

In [ ]:
# --- ASSERTIONS ---
date_cols = ['year', 'month', 'day_of_week', 'quarter', 'day_of_year', 'is_weekend', 'days_since_epoch', 'months_since_loan', 'season']
for col in date_cols:
    assert col in loans_dated.columns, f"Missing: {col}"

assert loans_dated['is_weekend'].dtype == bool
assert loans_dated['day_of_week'].between(0, 6).all()
assert loans_dated['quarter'].between(1, 4).all()
assert loans_dated['season'].isin(['Winter', 'Spring', 'Summer', 'Fall']).all()
assert (loans_dated['months_since_loan'] >= 0).all(), "All loans should predate 2023"
assert loans_dated['days_since_epoch'].min() == 0, "First row should have 0 days since epoch"

print("✓ Exercise 6 passed")

---
## Exercise 7 — Feature Selection: Variance & Correlation Filter

**Task:** Implement a basic feature selection pipeline — a common preprocessing step before modeling.

1. From the numeric columns of `loans`, remove:
   - **Near-zero variance** features: those with standard deviation < 0.01 (after standard scaling).
   - **Highly correlated** features: when two features have `|correlation| > 0.85`, keep the one with higher correlation to `target`, drop the other.
2. Return `selected_features`: list of column names that survive both filters.
3. Return `dropped_low_var`: list of columns dropped for low variance.
4. Return `dropped_high_corr`: list of columns dropped for high correlation.

In [ ]:
def select_features(df: pd.DataFrame, target_col: str = 'target'):
    """
    Returns (selected_features, dropped_low_var, dropped_high_corr).
    """
    # YOUR CODE HERE
    pass

selected_features, dropped_low_var, dropped_high_corr = select_features(loans)

In [ ]:
# --- ASSERTIONS ---
assert isinstance(selected_features, list)
assert isinstance(dropped_low_var, list)
assert isinstance(dropped_high_corr, list)
assert 'target' in selected_features, "Target must be in selected features"

# Dropped sets should not overlap
assert len(set(dropped_low_var) & set(dropped_high_corr)) == 0, "A feature shouldn't be in both drop lists"

# Check high-corr pairs among survivors are below threshold
numeric = loans[selected_features].select_dtypes(include='number')
corr = numeric.corr().abs()
np.fill_diagonal(corr.values, 0)
assert corr.max().max() <= 0.85 + 1e-6, "Surviving features must not have correlations > 0.85"

print(f"✓ Exercise 7 passed")
print(f"Selected: {len(selected_features)} | Low-var dropped: {dropped_low_var} | High-corr dropped: {dropped_high_corr}")